In [ ]:
import os 
import shutil
import torch
import yaml
import random
from ultralytics import YOLO
from datetime import datetime
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import cv2

# Используем относительные пути для работы внутри Docker
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "yolo_dataset" / "rgb"
BASE_DIR = PROJECT_ROOT / "results"

MODEL_SAVE_DIR = BASE_DIR / "models"
RUNS_DIR = BASE_DIR / "runs"
MODEL_PATH = "yolo26m-seg.pt"

AGRICULTURE_CLASSES = [
    'double_plant', 'drydown', 'endrow', 'nutrient_deficiency', 
    'planter_skip', 'storm_damage', 'water', 'waterway', 'weed_cluster'
]

def create_data_yaml(base_path: Path, output_path: Path) -> str:
    """Создаёт data.yaml с использованием абсолютных путей внутри контейнера"""

    train_path = (base_path / "images" / "train").resolve()
    val_path = (base_path / "images" / "val").resolve()

    if not train_path.exists():
        print(f"Внимание: Не найдены изображения для тренировки по пути {train_path}")
    
    new_cfg = {
        "path": str(base_path.resolve()),
        "train": str(train_path),      
        "val": str(val_path),          
        "nc": len(AGRICULTURE_CLASSES),
        "names": {i: name for i, name in enumerate(AGRICULTURE_CLASSES)}
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        yaml.dump(new_cfg, f, default_flow_style=False, sort_keys=False)

    print(f"Создан конфиг датасета: {output_path}")
    return str(output_path.resolve())
    

def train_yolo_model(epochs=50, batch_size=16, img_size=640, lr0=0.01, lrf=0.01):
    MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    RUNS_DIR.mkdir(parents=True, exist_ok=True)

    data_yaml_path = create_data_yaml(DATASET_ROOT, BASE_DIR / 'data.yaml')

    num_gpus = torch.cuda.device_count()
    device = list(range(num_gpus)) if num_gpus > 1 else '0' if num_gpus == 1 else 'cpu'
    print(f"Запуск обучения на устройствах: {device}")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f'train_{timestamp}'

    model = YOLO(MODEL_PATH, verbose=True)

    results = model.train(
        data=data_yaml_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=img_size,
        augment=True,
        patience=15,
        save=True,
        device=device,
        project=str(RUNS_DIR),
        name=run_name,
        lr0=lr0,
        lrf=lrf,
        plots=True,
        mask_ratio=1,
        overlap_mask=False, # Запрещает маскам разных классов неконтролируемо "наползать" друг на друга
        box=8.5,           # Увеличиваем фокус на точность локализации рамок (дефолт: 7.5)
        cls=2.0,           # Сильно увеличиваем штраф за ошибку в классификации (дефолт: 0.5)
        save_period=5,
        hsv_h=0.01,
        hsv_s=0.2,
        hsv_v=0.2,
        degrees=180,
        translate=0.1,
        scale=0.1,
        shear=0.0,
        perspective=0.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=0.0,
        mixup=0.0,
        cutmix=0.0,
        copy_paste=0.0,
        # copy_paste_mode=['flip', 'mixup'],
        copy_paste_mode='flip',
        amp=True,
        workers=8,
        cache=False
    )

    best_model_path = RUNS_DIR / run_name / 'weights' / 'best.pt'
    final_save = MODEL_SAVE_DIR / f'AgriYOLO_{timestamp}.pt'
    if best_model_path.exists():
        shutil.copy2(best_model_path, final_save)
        print(f"Лучшая модель сохранена: {final_save}")
    
    return model

def validate_model(model):
    num_gpus = torch.cuda.device_count()
    device = list(range(num_gpus)) if num_gpus > 1 else '0' if num_gpus == 1 else 'cpu'
    print(f"Запуск проверки на устройствах: {device}")   
    
    results = model.val(
        data=BASE_DIR / 'data.yaml',
        split="val",
        project=str(RUNS_DIR),
        name="val_metrics",
        conf=0.25,
        iou=0.5,
        device=device,
        verbose=True,
        plots=True,
        save_json=True
    )

    def to_float(val):
        if val is None: return 0.0
        if isinstance(val, (int, float)): return float(val)
        if hasattr(val, "item"): return float(val.item())
        if isinstance(val, (list, tuple, np.ndarray)):
            arr = np.asarray(val)
            return float(np.mean(arr)) if arr.size > 0 else 0.0
        return float(val)

    mAP50 = to_float(results.seg.map50)
    mAP50_95 = to_float(results.seg.map)
    precision = to_float(results.seg.mp)
    recall = to_float(results.seg.mr)
    f1 = to_float(results.seg.f1)

    ap50_raw = results.seg.ap50
    if isinstance(ap50_raw, np.ndarray):
        ap_per_class = ap50_raw.flatten().tolist()
    elif hasattr(ap50_raw, "tolist"):
        ap_per_class = ap50_raw.tolist()
    else:
        ap_per_class = [to_float(ap50_raw)]

    ap_dict = {AGRICULTURE_CLASSES[i]: float(ap_per_class[i]) for i in range(min(len(AGRICULTURE_CLASSES), len(ap_per_class)))}

    micro_f1 = 2 * (precision * recall) / (precision + recall + 1e-6) if (precision + recall) > 0 else 0.0

    metrics = {
        "Mask_mAP50": mAP50,
        "Mask_mAP50_95": mAP50_95,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "Micro_F1": micro_f1,
        "AP_per_Class": ap_dict,
        "Validation_Complete": True
    }

    print("\n--- Итоговые метрики сегментации ---")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    return metrics

In [ ]:
if __name__ == '__main__':
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model = train_yolo_model(epochs=60, batch_size=64, img_size=512, lr0=0.00038, lrf=0.01)
    metrics = validate_model(model)
    
    print("--- Поиск обученной модели ---")
    # Автоматический выбор весов для теста
    trained_models = list(MODEL_SAVE_DIR.glob("*.pt"))
    if trained_models:
        # Берём самую последнюю сохраненную модель по времени изменения файла
        MODEL_TEST_PATH = max(trained_models, key=lambda p: p.stat().st_mtime)
        print(f"Найдена финальная модель: {MODEL_TEST_PATH.name}")
    else:
        # Если скрипт обучения ещё не перенёс модель в models, ищем лучший чекпоинт внутри runs
        best_weights = list(RUNS_DIR.rglob("best.pt"))
        if best_weights:
            MODEL_TEST_PATH = max(best_weights, key=lambda p: p.stat().st_mtime)
            print(f"Найдена промежуточная лучшая модель: {MODEL_TEST_PATH}")
        else:
            raise FileNotFoundError(
                "Не найдено обученных моделей. "
                "Убедитесь, что этап обучения (train) завершился успешно."
            )

    # Загружаем модель для инференса
    model = YOLO(MODEL_TEST_PATH)

    # Поиск тестовых изображений
    test_images_dir = DATASET_ROOT / "test" / "images"

    # Если папки test нет или она пуста, берём картинки из валидации (val) для проверки работоспособности
    if not test_images_dir.exists() or not any(test_images_dir.glob("*.jpg")):
        print(f"Папка test не найдена или пуста. Переключаемся на валидационную выборку...")
        test_images_dir = DATASET_ROOT / "images" / "val"

    test_image_paths = list(test_images_dir.glob("*.jpg"))
    if not test_image_paths:
        raise FileNotFoundError(f"✗ В папке {test_images_dir} нет .jpg изображений!")

    # Выбираем случайные 5 картинок для визуализации результатов
    NUM_SAMPLES = min(5, len(test_image_paths))
    sampled_images = random.sample(test_image_paths, NUM_SAMPLES)
    print(f"✓ Выбрано {NUM_SAMPLES} случайных изображений для теста.")

    # Запуск инференса и отрисовка масок в Jupyter
    # Строим сетку графиков: 1 колонка, NUM_SAMPLES строк
    fig, axes = plt.subplots(NUM_SAMPLES, 1, figsize=(15, 8 * NUM_SAMPLES))
    if NUM_SAMPLES == 1:
        axes = [axes] 

    print("\n--- Запуск предсказания модели ---")
    for idx, img_path in enumerate(sampled_images):
        # conf=0.25 (порог уверенности), iou=0.45 (порог пересечения рамок)
        results = model.predict(source=str(img_path), conf=0.25, iou=0.45, verbose=False)
        
        for result in results:
            # Метод result.plot() автоматически накладывает полупрозрачные маски сегментации,
            # рисует контуры полей, подписи классов и рамки на изображение (возвращает BGR-массив)
            annotated_img_bgr = result.plot(labels=True, boxes=True, masks=True)
            
            annotated_img_rgb = cv2.cvtColor(annotated_img_bgr, cv2.COLOR_BGR2RGB)
            
            axes[idx].imshow(annotated_img_rgb)
            axes[idx].set_title(
                f"Изображение: {img_path.name} | Обнаружено аномалий: {len(result.boxes)}", 
                fontsize=14, pad=10
            )
            axes[idx].axis('off') 
    plt.tight_layout()
    plt.show()

Создан конфиг датасета: /usr/app/results/data.yaml
Запуск обучения на устройствах: [0, 1, 2, 3]
Ultralytics 8.4.67 🚀 Python-3.11.13 torch-2.8.0+cu129 CUDA:0 (NVIDIA A100 80GB PCIe, 81153MiB)
                                                      CUDA:1 (NVIDIA A100 80GB PCIe, 81153MiB)
                                                      CUDA:2 (NVIDIA A100 80GB PCIe, 81153MiB)
                                                      CUDA:3 (NVIDIA A100 80GB PCIe, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=False, cutmix=0.3, data=/usr/app/results/data.yaml, degrees=180, deterministic=True, device=0,1,2,3, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.7, flipud=0.7, format=torchsc